# 04 · Polychromatic electric-field knots

**A self-contained, Colab-ready reproduction and exploration of the 2021 Physical Review Research paper.**

**Reference:** M. F. Ferrer-Garcia, A. D'Errico, H. Larocque, A. Sit and E. Karimi, *Polychromatic electric field knots*, Physical Review Research **3**, 033226 (2021). [Journal article](https://doi.org/10.1103/PhysRevResearch.3.033226) · [Paper and supplementary derivations](https://arxiv.org/pdf/2103.06958).

### Learning objectives

1. Understand the distinction between spatial polarization Möbius strips and locally time-knotted electric-field trajectories.
2. Implement the three vector-vortex modes of Eqs. (3)–(4).
3. Focus every chromatic component with its **own wavenumber**, using the Richards–Wolf five-integral method.
4. Reconstruct the three-dimensional trajectory traced by the *real* instantaneous electric-field tip.
5. Investigate frequency ratios, phases, amplitudes, numerical convergence, and candidate knot regions.
6. Optionally compute a knot determinant with pyknotid, with appropriate numerical caveats.

**Reproduction level:** The physical construction and the paper's principal frequency/amplitude examples are implemented. Exact numerical datasets and all computational parameters underlying published knot histograms were not available, so plotted knot percentages are **not** claimed as exact replications. This notebook prioritizes scientific transparency and editable parameters.

**Runtime:** CPU is sufficient. Run sequentially or change the settings cell and then rerun the solver and visualization cells.

## 1. Theory: a knot traced in time, at one spatial point

The optical electric field is a sum of three monochromatic focused fields,

$$
\mathbf e(\mathbf r,t)=
\sum_{j=1}^{3}\operatorname{Re}\left[
a_j\mathbf E_j(\mathbf r;\omega_j)e^{-i\omega_jt}\right].
$$

The **curve** is the path of this real vector in electric-field component space, while the observation point $\mathbf r$ remains fixed. It is **not** a spatial curve in $(x,y,z)$ and is distinct from the director ribbons in Tutorial 03.

If $\omega_j=m_j\omega_0$ with integer, coprime harmonics $m_j$, let $\tau=\omega_0t$. The trajectory closes after $\tau\in[0,2\pi]$.

The incident vector vortex of Eq. (3) is

$$
\mathbf V_{\ell,\delta}(\theta,\phi)=
\frac{h_{|\ell|}(\theta)}{\sqrt2}
\left[e^{i\ell\phi}\mathbf e_R+
e^{i\delta}e^{-i\ell\phi}\mathbf e_L\right],
$$

where $\mathbf e_R=(1,-i)^T/\sqrt2$, $\mathbf e_L=(1,+i)^T/\sqrt2$ use the explicit $e^{-i\omega t}$ convention. The Jones convention fixes any naming ambiguity.

For Eq. (4), the three modes are $(\ell,\delta)=(1,0),(-1,0),(-1,\pi)$. The paper's example uses relative **incident** amplitudes $(1,2,2)$ and either frequency ratios $(1,3,2)$ or $(1,3/2,5/2)$. The second is equivalently represented by coprime integer harmonics $(2,3,5)$.

### Independent vectorial focusing of each colour

For each chromatic field separately we use

$$
\mathbf E_j(\mathbf r)=
-\frac{ik_jf}{2\pi}
\int_0^\alpha\int_0^{2\pi}
\sqrt{\cos\theta}\,
\mathbf R(\theta,\phi)\,\mathbf E_{{\rm in},j}(\theta,\phi)
\,e^{ik_j\mathbf s\cdot\mathbf r}
\sin\theta\,d\phi\,d\theta,
$$

with $k_j=2\pi n m_j/\lambda_{\rm ref}$ and $\alpha=\arcsin(\mathrm{NA}/n)$. The pupil envelope is
$h_{|\ell|}(\theta)=q^{|\ell|}e^{-(q/\mathrm{fill})^2}$, $q=\sin\theta/\sin\alpha$.

We omit a *common chosen lens reference phase* in the solver; real dispersive lens phases can be added as chromatic phases in the settings. The five Bessel orders after exact azimuthal integration are $\ell,\ell\pm1,\ell\pm2$. The longitudinal integrals include the additional $\sin\theta$ from the lens rotation. See Tutorial 01 for the derivation.

In [ ]:
%pip -q install numpy scipy matplotlib plotly ipywidgets
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.integrate import simpson
from scipy.special import jv
from scipy.spatial import cKDTree
from functools import lru_cache, reduce
from math import gcd
from IPython.display import display, clear_output
import ipywidgets as widgets
print("NumPy", np.__version__)

## 2. Parameters to experiment with

Change this cell and rerun the cells below it. Lengths are expressed in the wavelength $\lambda_{\rm ref}$ corresponding to the **common base frequency** $\omega_0$. In the $(2,3,5)$ case, the lowest optical wavelength is $\lambda_{\rm ref}/2$.

The original article's exact computational pupil fill and all plotting settings are not specified in the core text; the defaults here are clearly identified as **illustrative**. The amplitudes and frequencies reproduce the published choices.

In [ ]:
# --- User controls ---
PRESET = "paper_132"          # "paper_132", "paper_235", or "custom"
CUSTOM_HARMONICS = (1, 3, 2)
NA, n_medium = 0.85, 1.0       # example NA; requires NA < n
LAMBDA_REF, F = 1.0, 1.0       # vacuum base wavelength and focal length (length units)
FILL = 1.0                    # LG pupil envelope filling parameter
Z = 0.0                       # axial position / lambda_ref
HALF_WIDTH, NXY = 1.0, 81     # focal-plane grid (odd NXY)
NTHETA, NRADIAL = 401, 401    # integration and interpolation samples
AMPLITUDES = np.array([1.0,2.0,2.0])  # Eq. (4) paper example
PHASES = np.array([0.,0.,0.])          # additional chromatic phases (radians)

presets = {"paper_132":(1,3,2),"paper_235":(2,3,5)}
harmonics = tuple(map(int,presets.get(PRESET,CUSTOM_HARMONICS)))
assert len(harmonics)==3 and min(harmonics)>0 and reduce(gcd,harmonics)==1
assert 0<NA<n_medium and FILL>0 and NXY>=5 and NXY%2==1
assert NTHETA>10 and NRADIAL>10
assert len(AMPLITUDES)==len(PHASES)==3
weights = np.asarray(AMPLITUDES,complex)*np.exp(1j*np.asarray(PHASES,float))
aperture = np.arcsin(NA/n_medium)
x = np.linspace(-HALF_WIDTH,HALF_WIDTH,NXY); y=x.copy()
X,Y=np.meshgrid(x,y,indexing="xy")
rho, azimuth=np.hypot(X,Y),np.arctan2(Y,X)
radial_nodes=np.linspace(0,float(rho.max()),NRADIAL)
theta=np.linspace(0,aperture,NTHETA)
print("Harmonics:",harmonics," Vacuum wavelengths:",[LAMBDA_REF/m for m in harmonics])

## 3. The numerical solver

The code builds a focused Jones basis for charges $+1$ and $-1$ at each of the three frequencies. It integrates the polar angle numerically after performing the azimuthal integration analytically. The output is complex128 with component order $(E_x,E_y,E_z)$.

The chromatic basis fields must be recomputed when any optical or spatial parameter is changed. Rerunning this definition cell resets the cache.

In [ ]:
R_JONES=np.array([1,-1j],complex)/np.sqrt(2)
L_JONES=np.array([1,+1j],complex)/np.sqrt(2)

def interpolate_radial(values):
    rr=rho.ravel()
    return (np.interp(rr,radial_nodes,values.real)
            +1j*np.interp(rr,radial_nodes,values.imag)).reshape(rho.shape)

@lru_cache(None)
def focused_basis(ell,harmonic):
    """Return focused unit-x and unit-y incident vortex modes: (2,3,Ny,Nx)."""
    k=2*np.pi*n_medium*harmonic/LAMBDA_REF
    T=theta[None,:]; U=radial_nodes[:,None]
    st=np.sin(T); ct=np.cos(T); q=st/np.sin(aperture)
    g=np.sqrt(ct)*st*q**abs(ell)*np.exp(-(q/FILL)**2)*np.exp(1j*k*Z*ct)
    arg=k*U*st
    def integrate(a): return simpson(a,x=theta,axis=1)
    b1=integrate(g*(1+ct)*jv(ell,arg))
    b2=integrate(g*(1-ct)*jv(ell+2,arg))
    b3=integrate(g*(1-ct)*jv(ell-2,arg))
    b4=integrate(g*st*jv(ell+1,arg)) # longitudinal sin(theta)
    b5=integrate(g*st*jv(ell-1,arg))
    fac=-np.pi*(1j**(ell+1))*F/(LAMBDA_REF/(n_medium*harmonic))
    flong=np.pi*(1j**ell)*F/(LAMBDA_REF/(n_medium*harmonic))
    I1=fac*interpolate_radial(b1)*np.exp(1j*ell*azimuth)
    I2=fac*interpolate_radial(b2)*np.exp(1j*(ell+2)*azimuth)
    I3=fac*interpolate_radial(b3)*np.exp(1j*(ell-2)*azimuth)
    I4=flong*interpolate_radial(b4)*np.exp(1j*(ell+1)*azimuth)
    I5=flong*interpolate_radial(b5)*np.exp(1j*(ell-1)*azimuth)
    Fx=np.stack(((2*I1+I2+I3)/2,(I2-I3)/(2j),I5-I4))
    Fy=np.stack(((I2-I3)/(2j),(2*I1-I2-I3)/2,1j*(I4+I5)))
    return np.stack((Fx,Fy))

def vector_vortex(ell,delta,harmonic):
    """Focus Eq. (3): opposite charges in circular Jones components."""
    Bp=focused_basis(ell,harmonic)
    Bm=focused_basis(-ell,harmonic)
    return (R_JONES[0]*Bp[0]+R_JONES[1]*Bp[1]
            +np.exp(1j*delta)*(L_JONES[0]*Bm[0]+L_JONES[1]*Bm[1]))/np.sqrt(2)

focused_basis.cache_clear()
modes=[(+1,0),(-1,0),(-1,np.pi)]
E_colors=np.stack([w*vector_vortex(ell,d,h)
    for (ell,d),h,w in zip(modes,harmonics,weights)],axis=0)
assert E_colors.shape==(3,3,NXY,NXY) and np.isfinite(E_colors).all()
print("E_colors shape:",E_colors.shape)

## 4. Inspect the chromatic fields

The summed background intensity shown here is a time-averaged squared-field proxy $\sum_j\|\mathbf E_j\|^2$, without cross-frequency terms. **A bright region does not prove that it contains knots.** We must inspect the *instantaneous* three-dimensional trajectories.

In [ ]:
I_modes=np.sum(abs(E_colors)**2,axis=1)
I_total=I_modes.sum(axis=0)
fig,axes=plt.subplots(1,4,figsize=(15,3.4),constrained_layout=True)
for j,ax in enumerate(axes):
    data=I_total if j==3 else I_modes[j]
    im=ax.imshow(data/max(I_total.max(),1e-30),origin="lower",
       extent=(x[0],x[-1],y[0],y[-1]),vmin=0,vmax=1,cmap="viridis")
    ax.set_title("All colors" if j==3 else f"Mode {j+1} (harmonic {harmonics[j]})")
    ax.set_xlabel("x / lambda_ref");ax.set_ylabel("y / lambda_ref");ax.set_aspect("equal")
fig.colorbar(im,ax=axes,shrink=.75,label="I / maximum total I")
plt.show()
for j in range(3):
    frac=np.sum(abs(E_colors[j,2])**2)/max(np.sum(I_modes[j]),1e-30)
    print(f"Longitudinal share of mode {j+1}: {frac:.4f}")

## 5. The field-tip trajectory and interactive explorer

The local time curve at a point $(x_i,y_j,z)$ is

$$
\mathbf C(\tau)=
\sum_{m=1}^3\operatorname{Re}\left[
\mathbf E_m(x_i,y_j,z)e^{-ih_m\tau}\right],
\qquad 0\le\tau\le 2\pi,
$$

where $h_m$ are the selected integer harmonics. The path is plotted in **electric-field component space** $(e_x,e_y,e_z)$, not in physical laboratory position space. The plot's coordinate normalization is graphical only; it preserves topology.

Use the index sliders to select a different focal point without recalculating diffraction.

In [ ]:
NT=700
tau=np.linspace(0,2*np.pi,NT+1)
temporal=np.exp(-1j*np.outer(np.array(harmonics),tau))

def local_curve(j,i):
    phasors=E_colors[:,:,j,i]              # color, component
    return np.real(np.einsum("mc,mt->tc",phasors,temporal))

def show_trajectory(j,i):
    curve=local_curve(j,i)
    scale=max(float(np.max(np.ptp(curve,axis=0))),1e-15)
    p=curve/scale
    fig=go.Figure()
    fig.add_trace(go.Scatter3d(x=p[:,0],y=p[:,1],z=p[:,2],mode="lines",
       line=dict(color=tau,colorscale="Turbo",width=7),name="tip trajectory"))
    fig.add_trace(go.Scatter3d(x=[p[0,0]],y=[p[0,1]],z=[p[0,2]],
       mode="markers",marker=dict(color="black",size=5),name="start/end"))
    fig.update_layout(title=f"Instantaneous E at (x,y)=({x[i]:.3f},{y[j]:.3f}) lambda_ref",
        scene=dict(aspectmode="data",xaxis_title="e_x (normalized)",
                   yaxis_title="e_y (normalized)",zaxis_title="e_z (normalized)"),
        height=620,margin=dict(l=0,r=0,b=0,t=45))
    fig.show()
    return curve

default_i=min(NXY-1,NXY//2+NXY//8)
default_j=min(NXY-1,NXY//2+NXY//8)
curve=show_trajectory(default_j,default_i)

In [ ]:
ix=widgets.IntSlider(value=default_i,min=0,max=NXY-1,step=1,
                     description="x pixel",continuous_update=False)
iy=widgets.IntSlider(value=default_j,min=0,max=NXY-1,step=1,
                     description="y pixel",continuous_update=False)
output=widgets.Output()
def redraw(change=None):
    with output:
        clear_output(wait=True)
        show_trajectory(iy.value,ix.value)
ix.observe(redraw,names="value");iy.observe(redraw,names="value")
display(widgets.VBox([ix,iy,output]))
redraw()

### 5.1. An analytical control: Figure 1 of the article

Before examining the tightly focused fields, reproduce the analytic **example curves** in Figure 1. The planar two-frequency case is

$$
\mathbf C_{\rm plane}(\tau)=
(\cos\tau+\sin2\tau,\;-\sin\tau+\cos2\tau,\;0),
$$

and the three-dimensional three-frequency example is

$$
\mathbf C_{\rm 3D}(\tau)=
(\sin3\tau,\;\cos\tau-2\cos2\tau,\;\sin\tau+2\sin2\tau).
$$

These curves are provided directly by the article. They validate the time-domain curve construction independently of the optical focusing implementation. The article's three-dimensional example is not the same as asserting every focal-plane trajectory is a nontrivial knot.

In [ ]:
t = np.linspace(0,2*np.pi,1601)
planar = np.column_stack((np.cos(t)+np.sin(2*t),
                         -np.sin(t)+np.cos(2*t),np.zeros_like(t)))
example_3d = np.column_stack((np.sin(3*t),
                 np.cos(t)-2*np.cos(2*t),np.sin(t)+2*np.sin(2*t)))
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=example_3d[:,0],y=example_3d[:,1],z=example_3d[:,2],
             mode="lines",line=dict(width=7,color=t,colorscale="Turbo"),
             name="Paper Fig. 1(b)"))
fig.update_layout(height=560,scene=dict(aspectmode="data",
    xaxis_title="E_x",yaxis_title="E_y",zaxis_title="E_z"),
    title="Analytical three-frequency trajectory from paper Fig. 1(b)")
fig.show()
fig2,ax=plt.subplots(figsize=(5,5))
ax.plot(planar[:,0],planar[:,1]);ax.set_aspect("equal");ax.set_xlabel("E_x")
ax.set_ylabel("E_y");ax.set_title("Planar two-frequency example, Fig. 1(a)")
plt.show()

## 6. Is it really a knot?

A three-dimensional closed path can look knotted while remaining an **unknot**. It can also be close to a self-intersection, where its numerical topology is unstable. To diagnose candidate trajectories we check period closure, local sampling and nonlocal strand separation. Then the optional pyknotid package can calculate a knot determinant (as used in the supplementary paper).

A reliably computed determinant larger than one establishes nontrivial knotting, but determinant one does **not** establish an unknot and the determinant does not uniquely identify every knot type. A full topological classification requires additional invariants and convergence checks.

In [ ]:
def curve_diagnostics(curve,skip=12):
    p=np.asarray(curve,float)
    closure=np.linalg.norm(p[0]-p[-1])
    p=p[:-1]
    edges=np.linalg.norm(np.roll(p,-1,axis=0)-p,axis=1)
    tree=cKDTree(p)
    d,inds=tree.query(p,k=min(len(p),max(28,skip+15)))
    near=np.inf
    for i in range(len(p)):
        for dist,j in zip(d[i,1:],inds[i,1:]):
            separation=min(abs(i-int(j)),len(p)-abs(i-int(j)))
            if separation>skip:
                near=min(near,dist)
                break
    return {"closure":float(closure),"length":float(edges.sum()),
            "mean_step":float(edges.mean()),"nearest_nonlocal":float(near),
            "separation_to_step":float(near/max(edges.mean(),1e-30))}
print("Sample curve diagnostics:")
print(curve_diagnostics(curve))

In [ ]:
# OPTIONAL knot determinant (external package may require environment fixes).
# Uncomment the next line if you want a topological invariant:
# %pip -q install pyknotid
try:
    from pyknotid.spacecurves import Knot
    kobj=Knot(curve[:-1],verbose=False)
    det=kobj.determinant()
    print("Knot determinant:",det)
    print("Only classify a well-resolved, embedded, closed curve.")
except ImportError:
    print("pyknotid is optional. Install it by uncommenting the line above.")
except Exception as e:
    print("Topological identification unavailable:",type(e).__name__,str(e)[:250])

## 7. Explore the focal plane (not yet a knot-density classification)

The next two maps quantify (a) nonplanarity of the local trajectory through the ratio of the smallest and largest singular values of the centered sampled curve, and (b) a distance-to-discretization diagnostic for nonlocal parts of the curve.

These **are not knot-type maps**. They help select promising points to investigate with full knot-invariant calculations and identify under-resolved candidates. Reproducing the paper's Figure 3 knot statistics requires independent invariants on a large, well-resolved sample, numerical self-crossing screening, matching physical parameters and reporting error rates.

In [ ]:
STRIDE=5
ixs=np.arange(0,NXY,STRIDE);iys=np.arange(0,NXY,STRIDE)
nonplanar=np.empty((len(iys),len(ixs)))
separation=np.empty_like(nonplanar)
for jj,j in enumerate(iys):
    for ii,i in enumerate(ixs):
        c=local_curve(j,i)
        sv=np.linalg.svd(c[:-1]-c[:-1].mean(axis=0),compute_uv=False)
        nonplanar[jj,ii]=sv[-1]/max(sv[0],1e-30)
        separation[jj,ii]=curve_diagnostics(c)["separation_to_step"]
fig,axs=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
extent=(x[ixs[0]],x[ixs[-1]],y[iys[0]],y[iys[-1]])
for ax,array,label in zip(axs,[nonplanar,separation],
             ["Nonplanarity, NOT knot type","Nonlocal separation / sample step"]):
    im=ax.imshow(array,origin="lower",extent=extent,cmap="viridis")
    ax.set_title(label);ax.set_xlabel("x / lambda_ref");ax.set_ylabel("y / lambda_ref")
    fig.colorbar(im,ax=ax)
plt.show()

## 8. Save reproducible fields and explore the physics

The NPZ export contains **all three complex chromatic focused fields**, not just a plotted snapshot. It can be loaded for different visualization or topology algorithms without repeating the diffraction integrals.

**Suggested experiments:** swap the two published harmonic presets; vary only one relative phase; suppress the first (radial) mode and inspect how longitudinal content changes; vary the aperture; move away from the focal plane by changing $z$; refine polar-angle quadrature and radial interpolation independently; and compare truly classified knot-density maps with Figures 3 and 4 of the article.

The source model excludes chromatic objective dispersion, material interfaces, polarization-dependent aberrations and all measurement noise. Relative chromatic phase is an experimental degree of freedom, not a cosmetic setting.

**Cite:** Ferrer-Garcia et al., Phys. Rev. Research 3, 033226 (2021), DOI 10.1103/PhysRevResearch.3.033226.

In [ ]:
from pathlib import Path
target=Path("polychromatic_electric_knots_fields.npz")
np.savez_compressed(target,E_colors=E_colors,x=x,y=y,z=Z,
    harmonics=np.array(harmonics),amplitudes=AMPLITUDES,phases=PHASES,
    NA=NA,n_medium=n_medium,lambda_ref=LAMBDA_REF,focal_length=F,
    fill=FILL,ntheta=NTHETA,nradial=NRADIAL,
    ell=np.array([q[0] for q in modes]),delta=np.array([q[1] for q in modes]))
print("Saved",target.resolve())
try:
    from google.colab import files
    files.download(str(target))
except ImportError:
    print("Outside Colab: file is available in the working directory.")